In [1]:
import os
import glob
import pandas as pd

# Define relative path from 'master_datasets' to 'data_log/data_eish'
# '..' steps up to the project root 'DEP_Proj'
data_dir = os.path.join("..", "data_log", "data_eish")

# Match all cleaned CSV files
csv_files = glob.glob(os.path.join(data_dir, "*_cleaned.csv"))

# Store dataframes
df_list = []

for file in csv_files:
    df = pd.read_csv(file)
    # Store source filename (e.g. 'imu_log_20260721_032000_cleaned.csv') to track data provenance
    df["source_file"] = os.path.basename(file)
    df_list.append(df)

# Combine all DataFrames vertically
master_df = pd.concat(df_list, ignore_index=True)

# Save the master dataset in the current directory (master_datasets)
output_path = "eish_master.csv"
master_df.to_csv(output_path, index=False)

print(f"Successfully combined {len(csv_files)} files into '{output_path}'.")
print(f"Master Dataset Shape: {master_df.shape}")

Successfully combined 3 files into 'eish_master.csv'.
Master Dataset Shape: (6498, 14)


data cleaning, inspect structure

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = "eish_master.csv"  # Update path if running outside the eishmeet folder
df = pd.read_csv(file_path)

# Display basic information
print("--- Initial Overview ---")
print(f"Dataset Shape: {df.shape}")
print("\n--- Data Types & Missing Values ---")
print(df.info())
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

df.head()

--- Initial Overview ---
Dataset Shape: (6498, 14)

--- Data Types & Missing Values ---
<class 'pandas.DataFrame'>
RangeIndex: 6498 entries, 0 to 6497
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Timestamp     6498 non-null   str    
 1   AccX          6498 non-null   float64
 2   AccY          6498 non-null   float64
 3   AccZ          6498 non-null   float64
 4   GyroX         6498 non-null   float64
 5   GyroY         6498 non-null   float64
 6   GyroZ         6498 non-null   float64
 7   AccX_smooth   6498 non-null   float64
 8   AccY_smooth   6498 non-null   float64
 9   AccZ_smooth   6498 non-null   float64
 10  GyroX_smooth  6498 non-null   float64
 11  GyroY_smooth  6498 non-null   float64
 12  GyroZ_smooth  6498 non-null   float64
 13  source_file   6498 non-null   str    
dtypes: float64(12), str(2)
memory usage: 710.8 KB
None

--- Missing Values Count ---
Timestamp       0
AccX            0
AccY  

,Timestamp,AccX,AccY,AccZ,GyroX,GyroY,GyroZ,AccX_smooth,AccY_smooth,AccZ_smooth,GyroX_smooth,GyroY_smooth,GyroZ_smooth,source_file
0,2026-07-21 03:20:21.508,0.29,0.11,0.99,-4.70,4.82,2.56,0.290000,0.110,0.990000,-4.700000,4.8200,2.560000,imu_log_20260721_032000_cleaned.csv
1,2026-07-21 03:20:21.509,0.57,-0.83,0.03,129.58,1.59,-57.50,0.430000,-0.360,0.510000,62.440000,3.2050,-27.470000,imu_log_20260721_032000_cleaned.csv
2,2026-07-21 03:20:21.556,0.69,-0.78,0.28,184.14,16.60,-34.48,0.516667,-0.500,0.433333,103.006667,7.6700,-29.806667,imu_log_20260721_032000_cleaned.csv
3,2026-07-21 03:20:21.614,0.67,-0.52,0.48,203.37,99.98,-78.55,0.555000,-0.505,0.445000,128.097500,30.7475,-41.992500,imu_log_20260721_032000_cleaned.csv
4,2026-07-21 03:20:21.616,0.61,-0.40,0.58,166.99,137.70,-85.33,0.566000,-0.484,0.472000,135.876000,52.1380,-50.660000,imu_log_20260721_032000_cleaned.csv


In [2]:
# 1. Remove duplicate rows (ignoring pure identical duplicates)
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} duplicate rows.")

# 2. Handle missing values
# Option A: Forward-fill missing sensor values (common for time-series IMU data)
# Option B: Drop rows with missing values if critical
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].ffill().bfill()

# Check remaining missing values
print(f"Remaining nulls: {df.isnull().sum().sum()}")

Removed 0 duplicate rows.
Remaining nulls: 0


In [3]:
# Assuming there is a timestamp column (e.g., 'timestamp', 'time', or 'date')
# Adjust column name if different in your dataset
timestamp_col = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]

if timestamp_col:
    col_name = timestamp_col[0]
    df[col_name] = pd.to_datetime(df[col_name])
    df = df.sort_values(by=col_name).reset_index(drop=True)
    print(f"Parsed and sorted by timestamp column: '{col_name}'")
else:
    print("No timestamp column detected; skipping time sorting.")

Parsed and sorted by timestamp column: 'Timestamp'


In [4]:
# Identify numeric columns excluding metadata like source_file
features = df.select_dtypes(include=[np.number]).columns

for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Clip extreme values to the upper and lower threshold limits
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("Outlier clipping complete using IQR method.")

Outlier clipping complete using IQR method.


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Select feature columns to scale (exclude categorical target labels if present)
feature_cols = df.select_dtypes(include=[np.number]).columns

df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Features successfully scaled.")
df_scaled.head()

Features successfully scaled.


,Timestamp,AccX,AccY,AccZ,GyroX,GyroY,GyroZ,AccX_smooth,AccY_smooth,AccZ_smooth,GyroX_smooth,GyroY_smooth,GyroZ_smooth,source_file
0,2026-07-21 03:20:21.508,0.384795,0.315700,1.101036,-0.087682,0.030599,0.018687,0.404525,0.332544,1.136723,-0.110284,0.039359,0.030374,imu_log_20260721_032000_cleaned.csv
1,2026-07-21 03:20:21.509,0.929000,-1.142123,-0.216878,1.845680,-0.009733,-0.779126,0.690518,-0.432949,0.459381,1.068589,0.014928,-0.449334,imu_log_20260721_032000_cleaned.csv
2,2026-07-21 03:20:21.556,1.162230,-1.064579,0.126328,2.008720,0.177692,-0.473338,0.867562,-0.660969,0.351195,1.780876,0.082473,-0.486661,imu_log_20260721_032000_cleaned.csv
3,2026-07-21 03:20:21.614,1.123359,-0.661351,0.400894,2.008720,1.218833,-1.058746,0.945869,-0.669112,0.367658,2.016222,0.431583,-0.681321,imu_log_20260721_032000_cleaned.csv
4,2026-07-21 03:20:21.616,1.006743,-0.475246,0.538177,2.008720,1.689831,-1.148809,0.968340,-0.634909,0.405759,2.016222,0.755171,-0.819778,imu_log_20260721_032000_cleaned.csv


In [6]:
output_cleaned_path = "eish_master_preprocessed.csv"
df_scaled.to_csv(output_cleaned_path, index=False)

print(f"Preprocessed dataset successfully saved to: {output_cleaned_path}")
print(f"Final Processed Shape: {df_scaled.shape}")

Preprocessed dataset successfully saved to: eish_master_preprocessed.csv
Final Processed Shape: (6498, 14)
